In [1]:
import sys

sys.path.append('../../')

In [2]:
from ml_factory.datasets.precompute_tokens import precompute_tokens
from ml_factory.utils import merge_parts_to_dir
from ml_factory import DATA_RAW_DIR, DATA_PROCESSED_DIR
from transformers import AutoTokenizer
import torch
from pathlib import Path
from ml_factory.datasets import PromptBERTDataset
from torch.utils.data import Subset, DataLoader
from ml_factory.datasets.sampler import SplitSampler
import numpy as np
import torch

from torch.utils.data import Subset, DataLoader
from transformers import AutoModelForSequenceClassification
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from ml_factory.datasets.test import build_parquet_file

In [12]:
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

In [4]:
odd_file = build_parquet_file(
    DATA_RAW_DIR / "odd_data.parquet"
)

print(odd_file)

c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\raw\odd_data.parquet


In [8]:
data_path = DATA_RAW_DIR / 'odd_data.parquet'
output_path = DATA_PROCESSED_DIR / '32_128_processed_odd'

paths = precompute_tokens(dataset_path=data_path, 
                          out_path=output_path, 
                          tokenizer=AutoTokenizer,
                          pretrained_tokenizer="google/bert_uncased_L-2_H-128_A-2",
                          save_bytes=10_000_000,
                          token_chunk=True, 
                          stride=32, 
                          max_length=128)

Wrote 9880 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd.part0.npz
Wrote 10059 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd.part1.npz
Wrote 10050 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd.part2.npz
Wrote 9928 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd.part3.npz
Wrote 10041 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd.part4.npz
Wrote 10024 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd.part5.npz
Wrote 9897 i

In [9]:
merge_parts_to_dir(part_paths=paths,
                   out_dir=output_path)

Merged 8 parts into directory c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed_odd


WindowsPath('c:/Users/Jivesh Gawde/Documents/NMIMS/sem3/FortexAI/Backend/ml_factory/notebooks/../../ml_factory/data/processed/32_128_processed_odd')

In [10]:
dataset = PromptBERTDataset(output_path)

In [11]:
odd_loader = DataLoader(
    dataset,
    batch_size=100,
    shuffle=False
)

print("Number of odd-data batches:", len(odd_loader))

Number of odd-data batches: 745


In [13]:
save_dir = DATA_PROCESSED_DIR / "ensemble_models"

save_dir.mkdir(parents=True, exist_ok=True)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_models = []

for model_idx in range(1, 6):

    model_path = save_dir / f"bert_tiny_ensemble_model_{model_idx}.pt"

    checkpoint = torch.load(
        model_path,
        map_location=device,
        weights_only=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    loaded_models.append(model)

    print(
        f"Loaded Model {model_idx} | "
        f"Best epoch: {checkpoint['best_epoch']} | "
        f"Best val F1: {checkpoint['best_val_f1']:.4f}"
    )

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Model 1 | Best epoch: 5 | Best val F1: 0.9597


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Model 4 | Best epoch: 5 | Best val F1: 0.9597


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Model 5 | Best epoch: 5 | Best val F1: 0.9591


In [15]:
import numpy as np
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

all_model_probabilities = []
odd_labels = None


for model_idx, model in enumerate(loaded_models):

    model.eval()

    model_probabilities = []
    model_labels = []


    with torch.no_grad():

        for batch in odd_loader:

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)


            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


            # Probability of class 1
            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            )[:, 1]


            model_probabilities.extend(
                probabilities.cpu().numpy()
            )

            model_labels.extend(
                labels.cpu().numpy()
            )


    all_model_probabilities.append(
        np.array(model_probabilities)
    )


    if odd_labels is None:
        odd_labels = np.array(model_labels)


    print(
        f"Model {model_idx + 1}: "
        f"{len(model_probabilities)} predictions"
    )

Model 1: 74429 predictions
Model 2: 74429 predictions
Model 3: 74429 predictions
Model 4: 74429 predictions
Model 5: 74429 predictions


In [16]:
model_probabilities_matrix = np.vstack(
    all_model_probabilities
)

print(
    "Probability matrix shape:",
    model_probabilities_matrix.shape
)

Probability matrix shape: (5, 74429)


In [17]:
ensemble_probabilities = np.mean(
    model_probabilities_matrix,
    axis=0
)

In [18]:
ensemble_predictions = (
    ensemble_probabilities >= 0.5
).astype(int)

In [19]:

odd_accuracy = accuracy_score(
    odd_labels,
    ensemble_predictions
)

odd_precision = precision_score(
    odd_labels,
    ensemble_predictions,
    zero_division=0
)

odd_recall = recall_score(
    odd_labels,
    ensemble_predictions,
    zero_division=0
)

odd_f1 = f1_score(
    odd_labels,
    ensemble_predictions,
    zero_division=0
)

odd_roc_auc = roc_auc_score(
    odd_labels,
    ensemble_probabilities
)

odd_pr_auc = average_precision_score(
    odd_labels,
    ensemble_probabilities
)


print("\nEnsemble Odd-Data Results")
print("-------------------------")
print(f"Accuracy : {odd_accuracy:.4f}")
print(f"Precision: {odd_precision:.4f}")
print(f"Recall   : {odd_recall:.4f}")
print(f"F1       : {odd_f1:.4f}")
print(f"ROC-AUC  : {odd_roc_auc:.4f}")
print(f"PR-AUC   : {odd_pr_auc:.4f}")


Ensemble Odd-Data Results
-------------------------
Accuracy : 0.8367
Precision: 0.8850
Recall   : 0.7674
F1       : 0.8220
ROC-AUC  : 0.9113
PR-AUC   : 0.9158
